<a href="https://colab.research.google.com/github/gunnsmart/science-skills/blob/arena%2F01a0b805-science-skills/Image_Upscaler_%26_QC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Open-Source Topaz-like Image Upscaling Pipeline + Adobe Stock QC

**Architecture**

`Input Image → Image Analyzer → Model Router → Candidate Upscaling → Quality Evaluation → Artifact Detection → Best Output → Report`

This notebook is designed for Google Colab Free first. It uses open-source super-resolution models and a rule-based router. It does **not** claim to be equivalent to Topaz; it is a modular open-source pipeline inspired by multi-stage commercial workflows.

## Models in v1

| Model | Role | Code / loader | Weights | License notes |
|---|---|---|---|---|
| HAT | high-detail SR candidate | Spandrel loader | HF mirror of HAT checkpoints | Check original HAT repo/license and mirrored weight terms before commercial use. |
| Real-HAT | real-world GAN HAT candidate | Spandrel loader | HF mirror of Real_HAT_GAN_SRx4 | 4× only; review original and mirror licenses. |
| SwinIR | natural-photo SR candidate | Official SwinIR architecture | Official GitHub release x4 real-world checkpoint | SwinIR code is Apache-2.0; verify checkpoint terms for your use case. |
| Real-ESRGAN | degraded/compressed SR candidate | Internal BasicSR-free PyTorch RRDBNet | Official Real-ESRGAN GitHub releases x2/x4 | Real-ESRGAN project/license and pretrained weights are separate considerations. |

## Adobe Stock considerations

The pipeline prioritizes fidelity over invented detail. It detects and warns about over-sharpening, halos/ringing, repeated/fake texture, color artifacts, face/text risks, and excessive edge changes.  
**Technical QC only — platform acceptance is not guaranteed.**

## Usage

1. Run **CELL 1–4** once to set up the environment, analyzer, models, router, and runners.
2. Run **CELL 5** to upload one image, choose `AUTO` or `BENCHMARK`, choose 2× or 4×, and start the pipeline.
3. Run **CELL 6** to preview, export reports, create ZIP, and clean up.

## Limitations

- No single no-reference metric is sufficient; the final score combines multiple metrics and artifact penalties.
- MUSIQ/NIQE/BRISQUE depend on `pyiqa`; the notebook continues with fallback metrics if unavailable.
- LPIPS is computed only when `lpips` installs successfully and images can be aligned.
- Face/text artifact detection uses conservative CV heuristics, not a guarantee of semantic correctness.
- HAT/Real-HAT depend on Hugging Face mirrors because many official checkpoints are distributed via Drive/Baidu.
- Models are loaded one at a time and released after inference; this is slower but safer for Colab Free.

## Security note

PyTorch `.pth` checkpoints can be unsafe if they come from untrusted sources. This notebook uses official release URLs or explicitly listed Hugging Face mirrors and loads direct PyTorch checkpoints with `weights_only=True` where possible. Review model and weight sources before commercial or sensitive use.


In [ ]:
# CELL 1 — Setup, GPU, Dependencies, Registry
#@title CELL 1 — Setup, GPU, Dependencies, Registry


# -------------------- merged from old cell 1 --------------------
import os, sys, gc, json, math, time, shutil, subprocess, platform, warnings, hashlib, importlib.util, urllib.request
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings('ignore')
SEED = 1234
WORK_DIR = Path('/content/open_sr_pipeline')
INPUT_DIR = WORK_DIR / 'input'
OUTPUT_DIR = WORK_DIR / 'output'
CACHE_DIR = WORK_DIR / 'cache'
MODEL_DIR = WORK_DIR / 'models'
REPORT_DIR = OUTPUT_DIR
for d in [WORK_DIR, INPUT_DIR, OUTPUT_DIR, CACHE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tif', '.tiff'}

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', WORK_DIR)
print('Output directory:', OUTPUT_DIR)
print('Notebook generated for open-source SR router pipeline.')


# -------------------- merged from old cell 2 --------------------
HARDWARE = {
    'cuda': False,
    'gpu_name': 'CPU',
    'vram_total_gb': 0.0,
    'torch': None,
}
try:
    import torch
    HARDWARE['torch'] = torch.__version__
    HARDWARE['cuda'] = bool(torch.cuda.is_available())
    if HARDWARE['cuda']:
        idx = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(idx)
        HARDWARE['gpu_name'] = props.name
        HARDWARE['vram_total_gb'] = round(props.total_memory / (1024**3), 2)
except Exception as e:
    print('Torch/GPU detection warning:', e)

def default_tile_size(vram_gb=None):
    vram_gb = HARDWARE.get('vram_total_gb', 0) if vram_gb is None else vram_gb
    if not HARDWARE.get('cuda'):
        return 192
    if vram_gb >= 20:
        return 768
    if vram_gb >= 14:
        return 512
    if vram_gb >= 8:
        return 384
    return 256

def clear_memory(aggressive=True):
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            if aggressive and hasattr(torch.cuda, 'ipc_collect'):
                torch.cuda.ipc_collect()
    except Exception:
        pass

AUTO_TILE_SIZE = default_tile_size()
AUTO_TILE_PAD = 24 if HARDWARE.get('cuda') else 12
AUTO_BATCH_SIZE = 1
print(json.dumps(HARDWARE, indent=2))
print('Auto defaults:', {'tile_size': AUTO_TILE_SIZE, 'tile_pad': AUTO_TILE_PAD, 'batch_size': AUTO_BATCH_SIZE})


# -------------------- merged from old cell 3 --------------------
import subprocess, sys, importlib.util

def pip_install(packages, extra_args='', quiet=False):
    cmd = [sys.executable, '-m', 'pip', 'install']
    if extra_args:
        cmd.extend(extra_args.split())
    cmd.extend(packages.split() if isinstance(packages, str) else packages)
    print('Installing:', ' '.join(cmd[3:]))
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if not quiet or res.returncode != 0:
        print(res.stdout[-3000:])
    if res.returncode != 0:
        raise RuntimeError('pip install failed: ' + ' '.join(cmd))

def ensure_package(import_name, pip_name=None, extra_args='', optional=False):
    if importlib.util.find_spec(import_name) is not None:
        return True
    try:
        pip_install(pip_name or import_name, extra_args=extra_args)
        return importlib.util.find_spec(import_name) is not None
    except Exception as e:
        msg = f'Package {pip_name or import_name} unavailable: {e}'
        if optional:
            print('⚠️', msg)
            return False
        raise RuntimeError(msg)

CORE_PACKAGES = 'numpy pillow opencv-python-headless scikit-image pandas matplotlib tqdm requests huggingface_hub safetensors'
pip_install(CORE_PACKAGES, quiet=True)
PYIQA_AVAILABLE = ensure_package('pyiqa', 'pyiqa', optional=True)
LPIPS_AVAILABLE = ensure_package('lpips', 'lpips', optional=True)

# Spandrel is used lazily by HAT/Real-HAT. Try normal install first, then ignore Python metadata if Colab image is newer.
def ensure_spandrel(extra_arches=False):
    if importlib.util.find_spec('spandrel') is None:
        try:
            pip_install('spandrel')
        except Exception:
            pip_install('spandrel==0.4.2', extra_args='--ignore-requires-python')
    if extra_arches and importlib.util.find_spec('spandrel_extra_arches') is None:
        try:
            pip_install('spandrel-extra-arches')
        except Exception:
            pip_install('spandrel-extra-arches', extra_args='--ignore-requires-python')
    if extra_arches:
        import spandrel_extra_arches
        spandrel_extra_arches.install()
    return True

print('Dependency status:', {'pyiqa': PYIQA_AVAILABLE, 'lpips': LPIPS_AVAILABLE})


# -------------------- merged from old cell 4 --------------------
import hashlib, json, urllib.request
from pathlib import Path

MODEL_REGISTRY = {
    'HAT': {
        'family': 'Transformer SR',
        'status': 'available_if_downloadable',
        'implementation': 'spandrel_hf',
        'scales': [2, 4],
        'hf_repo': 'jaideepsingh/upscale_models',
        'hf_files': {2: 'HAT/HAT-L_SRx2_ImageNet-pretrain.pth', 4: 'HAT/HAT-L_SRx4_ImageNet-pretrain.pth'},
        'code_license': 'HAT official repo; review upstream license',
        'weight_license': 'Hugging Face mirror; verify original and mirror terms before commercial use',
        'provenance': {'source_type': 'mirror', 'mirror': 'Hugging Face', 'official_repo': 'https://github.com/XPixelGroup/HAT', 'sha256': None, 'validation': 'hf_hub_download + existence/min-size; SHA256 not independently verified'},
        'best_for': 'high detail, low noise, structured texture',
    },
    'Real-HAT': {
        'family': 'Transformer/GAN SR',
        'status': 'available_if_downloadable',
        'implementation': 'spandrel_hf',
        'scales': [4],
        'hf_repo': 'jaideepsingh/upscale_models',
        'hf_files': {4: 'HAT/Real_HAT_GAN_SRx4.pth'},
        'code_license': 'HAT official repo; review upstream license',
        'weight_license': 'Hugging Face mirror; verify original and mirror terms before commercial use',
        'provenance': {'source_type': 'mirror', 'mirror': 'Hugging Face', 'official_repo': 'https://github.com/XPixelGroup/HAT', 'sha256': None, 'validation': 'hf_hub_download + existence/min-size; SHA256 not independently verified'},
        'best_for': 'real-world detail at 4x; may hallucinate more than conservative models',
    },
    'SwinIR': {
        'family': 'Transformer SR',
        'status': 'available_if_downloadable',
        'implementation': 'swinir_official',
        'scales': [2, 4],
        'native_scales': [4],
        'repo': 'https://github.com/JingyunLiang/SwinIR',
        'arch_url': 'https://raw.githubusercontent.com/JingyunLiang/SwinIR/main/models/network_swinir.py',
        'weights': {4: 'https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth'},
        'code_license': 'Apache-2.0 for SwinIR code',
        'weight_license': 'Official project release; verify terms before commercial use',
        'provenance': {'source_type': 'official_release', 'repo': 'https://github.com/JingyunLiang/SwinIR', 'sha256': None, 'validation': 'direct HTTPS download + min-size; SHA256 not independently verified'},
        'best_for': 'natural photos, balanced fidelity/detail',
    },
    'Real-ESRGAN': {
        'family': 'GAN SR',
        'status': 'available_if_downloadable',
        'implementation': 'internal_rrdb',
        'scales': [2, 4],
        'weights': {
            2: 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
            4: 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        },
        'expected_bytes': {2: 67061725, 4: 67040989},
        'code_license': 'Real-ESRGAN project license; this notebook uses an internal PyTorch RRDB inference implementation',
        'weight_license': 'Official Real-ESRGAN release weights; verify terms before commercial use',
        'provenance': {'source_type': 'official_release', 'repo': 'https://github.com/xinntao/Real-ESRGAN', 'sha256': None, 'validation': 'direct HTTPS download + exact byte-size check'},
        'best_for': 'degraded, noisy, compressed or low-quality photos',
    },
}

PREFETCH_MODELS = False #@param {type:"boolean"}
PREFETCH_SCALE = "4x" #@param ["2x", "4x"]

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def validate_downloaded_file(dst, expected_bytes=None, min_bytes=1024*1024, sha256=None):
    dst = Path(dst)
    if not dst.exists():
        return False, 'missing'
    size = dst.stat().st_size
    if expected_bytes and size != expected_bytes:
        return False, f'size mismatch: got {size}, expected {expected_bytes}'
    if not expected_bytes and size < min_bytes:
        return False, f'file too small: {size} bytes < {min_bytes}'
    if sha256:
        actual = sha256_file(dst)
        if actual.lower() != sha256.lower():
            return False, f'sha256 mismatch: got {actual}, expected {sha256}'
    return True, 'ok'


def download_url(url, dst, expected_bytes=None, min_bytes=1024*1024, sha256=None, retries=3, backoff=2.0):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    ok, reason = validate_downloaded_file(dst, expected_bytes, min_bytes, sha256)
    if ok:
        return dst
    if dst.exists():
        print(f'Existing file failed validation ({reason}); re-downloading: {dst}')
        dst.unlink(missing_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            tmp = dst.with_suffix(dst.suffix + '.partial')
            tmp.unlink(missing_ok=True)
            print(f'Downloading ({attempt}/{retries}): {url}')
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r, open(tmp, 'wb') as f:
                while True:
                    chunk = r.read(1024*1024)
                    if not chunk: break
                    f.write(chunk)
            tmp.replace(dst)
            ok, reason = validate_downloaded_file(dst, expected_bytes, min_bytes, sha256)
            if ok:
                return dst
            raise RuntimeError(reason)
        except Exception as e:
            last_error = e
            dst.unlink(missing_ok=True)
            try:
                tmp.unlink(missing_ok=True)
            except Exception:
                pass
            print(f'⚠️ Download/validation failed for {dst.name}: {e}')
            if attempt < retries:
                time.sleep(backoff ** (attempt - 1))
    raise RuntimeError(f'Failed to download/validate {dst.name} after {retries} attempts: {last_error}')

def hf_download(repo_id, filename, min_bytes=1024*1024):
    from huggingface_hub import hf_hub_download
    path = Path(hf_hub_download(repo_id=repo_id, filename=filename, cache_dir=str(CACHE_DIR/'hf')))
    ok, reason = validate_downloaded_file(path, expected_bytes=None, min_bytes=min_bytes, sha256=None)
    if not ok:
        raise RuntimeError(f'Hugging Face file validation failed for {repo_id}/{filename}: {reason}')
    return path

print('Model registry:')
for name, meta in MODEL_REGISTRY.items():
    print(f"- {name}: {meta['status']} | scales={meta['scales']} | {meta['best_for']}")
    print('  code license:', meta['code_license'])
    print('  weights:', meta['weight_license'])

if PREFETCH_MODELS:
    scale = int(PREFETCH_SCALE.replace('x',''))
    for name, meta in MODEL_REGISTRY.items():
        try:
            if scale not in meta['scales']:
                print(f'Skip {name}: no {scale}x checkpoint')
                continue
            if meta['implementation'] == 'spandrel_hf':
                path = hf_download(meta['hf_repo'], meta['hf_files'][scale])
            elif meta['implementation'] == 'internal_rrdb':
                path = download_url(meta['weights'][scale], MODEL_DIR/f"{name}_{scale}x.pth", meta.get('expected_bytes',{}).get(scale))
            elif meta['implementation'] == 'swinir_official':
                native = 4
                path = download_url(meta['weights'][native], MODEL_DIR/'SwinIR_x4.pth')
            print('Ready:', name, path)
        except Exception as e:
            print(f'⚠️ {name} prefetch failed:', e)
else:
    print('Prefetch disabled. Models download lazily when selected.')


# -------------------- merged from old cell 5 --------------------
import os, sys, time, json, math, gc, copy, shutil, traceback, tempfile
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import cv2
from PIL import Image, ImageOps, ImageDraw, ImageFont
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric, peak_signal_noise_ratio as psnr_metric
from IPython.display import display, Markdown
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError('PyTorch is required for this notebook.') from e

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP_DEFAULT = bool(torch.cuda.is_available())

@dataclass
class CandidateResult:
    model: str
    output_path: str
    selected: bool
    scale: int
    settings: dict
    analysis: dict
    metrics: dict
    artifacts: dict
    scores: dict
    processing_time: float
    gpu: str
    notes: list
    errors: list



def torch_load_weights_only(path, map_location='cpu'):
    # Prefer PyTorch's safer weights-only unpickler for .pth checkpoints.
    # Older torch versions may not support weights_only; in that case this
    # falls back only for compatibility and the report/license notes still warn
    # that .pth files must come from trusted official/mirrored sources.
    try:
        return torch.load(str(path), map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(str(path), map_location=map_location)

print('Imports ready. Device:', DEVICE)



In [ ]:
# CELL 2 — Image Analyzer, Quality Metrics, Artifact Detector
#@title CELL 2 — Image Analyzer, Quality Metrics, Artifact Detector


# -------------------- merged from old cell 6 --------------------
def load_image_rgb(path):
    path = Path(path)
    if path.suffix.lower() not in SUPPORTED_EXTS:
        raise ValueError(f'Unsupported image format {path.suffix}. Supported: {sorted(SUPPORTED_EXTS)}')
    try:
        img = Image.open(path)
        img = ImageOps.exif_transpose(img)
        has_alpha = img.mode in ('RGBA','LA') or ('transparency' in img.info)
        if img.mode == 'CMYK':
            print('⚠️ CMYK image detected; converting to RGB for analysis/inference.')
        rgb = img.convert('RGB')
        return rgb, {'mode': img.mode, 'has_alpha': has_alpha, 'format': img.format}
    except Exception as e:
        raise RuntimeError(f'Cannot read image. It may be corrupted or unsupported: {e}')

def laplacian_variance(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def estimate_noise(gray):
    med = cv2.medianBlur(gray, 3)
    noise = gray.astype(np.float32) - med.astype(np.float32)
    return float(np.std(noise))

def jpeg_block_score(gray):
    g = gray.astype(np.float32)
    if g.shape[0] < 16 or g.shape[1] < 16:
        return 0.0
    # Compare pixels across JPEG 8x8 block boundaries. For widths/heights that
    # are not multiples of 8, the two slices can have different lengths, so crop
    # to the shared count before subtracting. This prevents broadcast errors on
    # common screenshot sizes such as 1080x2400.
    v = 0.0
    if g.shape[1] > 8:
        right = g[:, 8::8]
        left = g[:, 7::8]
        n = min(right.shape[1], left.shape[1])
        if n > 0:
            v = float(np.mean(np.abs(right[:, :n] - left[:, :n])))
    h = 0.0
    if g.shape[0] > 8:
        below = g[8::8, :]
        above = g[7::8, :]
        n = min(below.shape[0], above.shape[0])
        if n > 0:
            h = float(np.mean(np.abs(below[:n, :] - above[:n, :])))
    baseline = np.mean(np.abs(np.diff(g, axis=1))) + np.mean(np.abs(np.diff(g, axis=0))) + 1e-6
    return float((v + h) / (2 * baseline))

def edge_density(gray):
    med = np.median(gray)
    lower = int(max(0, 0.66 * med))
    upper = int(min(255, 1.33 * med + 30))
    edges = cv2.Canny(gray, lower, upper)
    return float(np.mean(edges > 0)), edges

def texture_density(gray):
    f = np.fft.fftshift(np.fft.fft2(gray.astype(np.float32)))
    mag = np.log1p(np.abs(f))
    h, w = mag.shape
    cy, cx = h//2, w//2
    yy, xx = np.ogrid[:h, :w]
    r = np.sqrt((yy-cy)**2 + (xx-cx)**2)
    hi = mag[r > min(h,w)*0.18].mean() if np.any(r > min(h,w)*0.18) else 0
    lo = mag[r <= min(h,w)*0.18].mean() + 1e-6
    return float(hi / lo)

def brightness_contrast(rgb_np):
    lab = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2LAB)
    L = lab[:,:,0]
    return float(np.mean(L)/255.0), float(np.std(L)/255.0)

def image_complexity(gray, edge_d, tex_d):
    hist = cv2.calcHist([gray],[0],None,[64],[0,256]).ravel()
    p = hist / (hist.sum()+1e-9)
    entropy = float(-(p[p>0] * np.log2(p[p>0])).sum()/6.0)
    comp = 0.35*min(edge_d/0.18,1) + 0.35*min(tex_d/0.45,1) + 0.30*min(entropy,1)
    return float(comp), entropy

def analyze_image(path):
    rgb, info = load_image_rgb(path)
    rgb_np = np.array(rgb)
    gray = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    sharp = laplacian_variance(gray)
    noise = estimate_noise(gray)
    block = jpeg_block_score(gray)
    ed, edges = edge_density(gray)
    tex = texture_density(gray)
    bright, contrast = brightness_contrast(rgb_np)
    comp, entropy = image_complexity(gray, ed, tex)
    features = {
        'path': str(path),
        'resolution': {'width': int(w), 'height': int(h), 'megapixels': round(w*h/1e6, 4)},
        'aspect_ratio': round(w/h, 6) if h else None,
        'source_info': info,
        'sharpness_laplacian': round(sharp, 4),
        'noise_sigma': round(noise, 4),
        'jpeg_block_score': round(block, 4),
        'edge_density': round(ed, 6),
        'texture_density': round(tex, 6),
        'brightness': round(bright, 6),
        'contrast': round(contrast, 6),
        'entropy': round(entropy, 6),
        'complexity': round(comp, 6),
        'flags': {
            'low_resolution': bool(max(w,h) < 1200),
            'blurry': bool(sharp < 80),
            'noisy': bool(noise > 7.0),
            'compressed': bool(block > 1.18),
            'high_detail': bool(tex > 0.38 and ed > 0.055),
            'low_contrast': bool(contrast < 0.16),
            'very_dark_or_bright': bool(bright < 0.18 or bright > 0.88),
        }
    }
    return features

print('Image analyzer ready: resolution, blur, noise, compression, edges, texture, brightness, contrast, complexity.')


# -------------------- merged from old cell 11 --------------------
IQA_MODELS = {}
LPIPS_MODEL = None

def image_to_np_rgb(path, max_side=None):
    img, _ = load_image_rgb(path)
    if max_side and max(img.size) > max_side:
        r = max_side / max(img.size)
        img = img.resize((int(img.width*r), int(img.height*r)), Image.Resampling.LANCZOS)
    return np.array(img)

def get_pyiqa_metric(name):
    if not PYIQA_AVAILABLE:
        return None
    if name not in IQA_MODELS:
        try:
            import pyiqa
            IQA_MODELS[name] = pyiqa.create_metric(name, device=DEVICE)
        except Exception as e:
            print(f'⚠️ pyiqa metric {name} unavailable:', e)
            IQA_MODELS[name] = None
    return IQA_MODELS[name]

def compute_pyiqa(path, metric_name):
    metric = get_pyiqa_metric(metric_name)
    if metric is None:
        return None
    try:
        val = metric(str(path))
        if hasattr(val, 'detach'):
            val = val.detach().float().cpu().item()
        return float(val)
    except Exception as e:
        print(f'⚠️ {metric_name} failed:', e)
        return None

def get_lpips_model():
    global LPIPS_MODEL
    if not LPIPS_AVAILABLE:
        return None
    if LPIPS_MODEL is None:
        try:
            import lpips
            LPIPS_MODEL = lpips.LPIPS(net='alex').to(DEVICE).eval()
        except Exception as e:
            print('⚠️ LPIPS unavailable:', e)
            LPIPS_MODEL = None
    return LPIPS_MODEL

def compute_lpips_ref(original_path, output_path):
    model = get_lpips_model()
    if model is None:
        return None
    try:
        a = Image.open(original_path).convert('RGB')
        b = Image.open(output_path).convert('RGB').resize(a.size, Image.Resampling.LANCZOS)
        def to_t(img):
            arr = np.array(img).astype(np.float32)/127.5 - 1.0
            return torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            return float(model(to_t(a), to_t(b)).detach().cpu().item())
    except Exception as e:
        print('⚠️ LPIPS failed:', e)
        return None

def compute_basic_quality(path):
    rgb = image_to_np_rgb(path, max_side=1600)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    ed, _ = edge_density(gray)
    return {
        'sharpness': laplacian_variance(gray),
        'noise_sigma': estimate_noise(gray),
        'jpeg_block_score': jpeg_block_score(gray),
        'edge_density': ed,
        'texture_density': texture_density(gray),
        'brightness': brightness_contrast(rgb)[0],
        'contrast': brightness_contrast(rgb)[1],
    }

def compute_quality_metrics(original_path, output_path, ground_truth_path=None):
    metrics = compute_basic_quality(output_path)
    metrics.update({
        'MUSIQ': compute_pyiqa(output_path, 'musiq'),
        'NIQE': compute_pyiqa(output_path, 'niqe'),
        'BRISQUE': compute_pyiqa(output_path, 'brisque'),
        'LPIPS_to_input_resized': compute_lpips_ref(original_path, output_path),
    })
    if ground_truth_path:
        try:
            gt = image_to_np_rgb(ground_truth_path)
            out = Image.open(output_path).convert('RGB').resize((gt.shape[1], gt.shape[0]), Image.Resampling.LANCZOS)
            out_np = np.array(out)
            metrics['PSNR_GT'] = float(psnr_metric(gt, out_np, data_range=255))
            metrics['SSIM_GT'] = float(ssim_metric(gt, out_np, channel_axis=2, data_range=255))
        except Exception as e:
            metrics['GT_metric_error'] = str(e)
    else:
        metrics['PSNR_GT'] = None
        metrics['SSIM_GT'] = None
    return {k: (round(v,6) if isinstance(v, float) and math.isfinite(v) else v) for k,v in metrics.items()}

print('Quality metrics ready: MUSIQ, NIQE, BRISQUE when available; LPIPS optional; PSNR/SSIM with GT; sharpness/edges/artifacts baseline.')


# -------------------- merged from old cell 12 --------------------
def norm01(value, low, high, invert=False):
    if value is None or not math.isfinite(float(value)):
        return None
    x = (float(value)-low)/(high-low+1e-9)
    x = max(0.0, min(1.0, x))
    return 1.0-x if invert else x

def halo_score(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    edges = cv2.Canny(gray.astype(np.uint8), 80, 160)
    dil = cv2.dilate(edges, np.ones((5,5), np.uint8), iterations=1) > 0
    near = dil & ~(edges > 0)
    if near.sum() < 50:
        return 0.0
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    return float(np.mean(np.abs(lap[near])) / 40.0)

def ringing_score(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    sobel = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    edges = cv2.Canny(gray.astype(np.uint8), 100, 200) > 0
    if edges.sum() < 50: return 0.0
    osc = cv2.Laplacian(sobel, cv2.CV_32F)
    return float(np.mean(np.abs(osc[cv2.dilate(edges.astype(np.uint8), np.ones((7,7),np.uint8))>0])) / 80.0)

def repeated_texture_score(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h,w = gray.shape
    if h < 128 or w < 128: return 0.0
    small = cv2.resize(gray, (min(512,w), min(512,h)), interpolation=cv2.INTER_AREA)
    step = max(32, min(small.shape)//8)
    patches = []
    for y in range(0, small.shape[0]-step, step):
        for x in range(0, small.shape[1]-step, step):
            p = small[y:y+step, x:x+step].astype(np.float32)
            if p.std() > 8:
                p = (p - p.mean())/(p.std()+1e-6)
                patches.append(p.ravel())
    if len(patches) < 6: return 0.0
    P = np.stack(patches[:80])
    corr = np.corrcoef(P)
    upper = corr[np.triu_indices_from(corr, 1)]
    return float(np.mean(upper > 0.92))

def color_artifact_score(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    sat = hsv[:,:,1].astype(np.float32)/255
    clipped = np.mean((rgb <= 2) | (rgb >= 253))
    oversat = np.mean(sat > 0.96)
    return float(min(1.0, clipped*4 + oversat*2))

def text_like_risk(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 180)
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in cnts:
        x,y,w,h = cv2.boundingRect(c)
        if 5 <= w <= 160 and 5 <= h <= 80 and 0.15 <= w/max(h,1) <= 12:
            boxes.append((x,y,w,h))
    density = len(boxes) / max(1, (rgb.shape[0]*rgb.shape[1]/1e6))
    return float(min(1.0, density/120.0))

def face_risk(original_rgb, output_rgb):
    try:
        cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        detector = cv2.CascadeClassifier(cascade_path)
        if detector.empty(): return 0.0
        def detect(img):
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            return detector.detectMultiScale(gray, 1.1, 4)
        fo, fu = len(detect(original_rgb)), len(detect(output_rgb))
        if fo == 0 and fu == 0: return 0.0
        if fo > 0 and fu == 0: return 0.7
        return float(min(1.0, abs(fu-fo)/max(fo,1)*0.5))
    except Exception:
        return 0.0

def detect_artifacts(original_path, output_path):
    orig = image_to_np_rgb(original_path, max_side=1024)
    out = image_to_np_rgb(output_path, max_side=2048)
    out_for_compare = np.array(Image.fromarray(out).resize((orig.shape[1], orig.shape[0]), Image.Resampling.LANCZOS))
    orig_gray = cv2.cvtColor(orig, cv2.COLOR_RGB2GRAY)
    out_gray = cv2.cvtColor(out_for_compare, cv2.COLOR_RGB2GRAY)
    orig_sharp = laplacian_variance(orig_gray)
    out_sharp = laplacian_variance(out_gray)
    sharp_ratio = out_sharp/(orig_sharp+1e-6)
    orig_edge = edge_density(orig_gray)[0]
    out_edge = edge_density(out_gray)[0]
    edge_ratio = out_edge/(orig_edge+1e-6)
    scores = {
        'oversharpening': norm01(sharp_ratio, 2.0, 8.0) or 0.0,
        'ringing': min(1.0, ringing_score(out)),
        'halos': min(1.0, halo_score(out)),
        'fake_texture': min(1.0, repeated_texture_score(out) * 3.0),
        'repeated_texture': min(1.0, repeated_texture_score(out) * 3.0),
        'unnatural_edges': norm01(edge_ratio, 1.8, 5.0) or 0.0,
        'face_distortion_risk': face_risk(orig, out_for_compare),
        'text_distortion_risk': text_like_risk(out),
        'color_artifacts': color_artifact_score(out),
    }
    weighted = (0.15*scores['oversharpening'] + 0.13*scores['ringing'] + 0.13*scores['halos'] +
                0.14*scores['fake_texture'] + 0.12*scores['unnatural_edges'] + 0.10*scores['face_distortion_risk'] +
                0.10*scores['text_distortion_risk'] + 0.13*scores['color_artifacts'])
    warnings_list = [k for k,v in scores.items() if v >= 0.55]
    return {'scores': {k: round(float(v),6) for k,v in scores.items()}, 'artifact_score': round(float(min(1.0, weighted)),6), 'warnings': warnings_list}

print('Artifact detector ready: oversharpening, ringing, halos, fake/repeated texture, unnatural edges, face/text/color risks.')



In [ ]:
# CELL 3 — Upscaler Engines: Base, HAT, SwinIR, Real-ESRGAN
#@title CELL 3 — Upscaler Engines: Base, HAT, SwinIR, Real-ESRGAN


# -------------------- merged from old cell 7 --------------------
class UpscalerError(RuntimeError):
    pass

class BaseUpscaler:
    name = 'Base'
    supported_scales = [2,4]
    license_info = {}

    def __init__(self, scale=4, tile_size=None, tile_pad=None, batch_size=1, fp16=True):
        self.scale = int(scale)
        self.tile_size = int(tile_size or AUTO_TILE_SIZE)
        self.tile_pad = int(tile_pad if tile_pad is not None else AUTO_TILE_PAD)
        self.batch_size = int(batch_size or 1)
        self.fp16 = bool(fp16 and torch.cuda.is_available())
        self.device = DEVICE
        self.model = None
        self.notes = []
        if self.scale not in self.supported_scales:
            raise UpscalerError(f'{self.name} does not support requested {self.scale}x in this notebook. Supported: {self.supported_scales}')

    def load(self):
        raise NotImplementedError

    def unload(self):
        try:
            del self.model
        except Exception:
            pass
        self.model = None
        clear_memory(True)

    def upscale(self, input_path, output_path):
        raise NotImplementedError

    @staticmethod
    def pil_to_tensor(img):
        arr = np.array(img.convert('RGB')).astype(np.float32) / 255.0
        t = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0)
        return t

    @staticmethod
    def tensor_to_pil(t):
        t = t.detach().float().clamp(0,1).cpu()[0]
        arr = (t.permute(1,2,0).numpy() * 255.0).round().astype(np.uint8)
        return Image.fromarray(arr, 'RGB')

    def forward_tiled(self, model, x, scale=None):
        scale = int(scale or self.scale)
        _, c, h, w = x.shape
        tile = int(self.tile_size or 0)
        pad = int(self.tile_pad or 0)
        if tile <= 0 or (h <= tile and w <= tile):
            with torch.no_grad():
                return model(x)
        output = torch.zeros((1, c, h*scale, w*scale), dtype=x.dtype, device=x.device)
        weight = torch.zeros_like(output)
        stride = max(tile - 2*pad, 32)
        ys = list(range(0, h, stride))
        xs = list(range(0, w, stride))
        for y in ys:
            for x0 in xs:
                y0 = min(y, h-tile) if h > tile else 0
                x1 = min(x0, w-tile) if w > tile else 0
                y0 = max(0,y0); x1 = max(0,x1)
                y2 = min(y0+tile, h); x2 = min(x1+tile, w)
                py0 = max(0, y0-pad); px0 = max(0, x1-pad)
                py2 = min(h, y2+pad); px2 = min(w, x2+pad)
                patch = x[:,:,py0:py2,px0:px2]
                with torch.no_grad():
                    out_patch = model(patch)
                crop_y0 = (y0-py0)*scale; crop_x0 = (x1-px0)*scale
                crop_y2 = crop_y0 + (y2-y0)*scale; crop_x2 = crop_x0 + (x2-x1)*scale
                target_y0 = y0*scale; target_x0 = x1*scale
                target_y2 = y2*scale; target_x2 = x2*scale
                output[:,:,target_y0:target_y2,target_x0:target_x2] += out_patch[:,:,crop_y0:crop_y2,crop_x0:crop_x2]
                weight[:,:,target_y0:target_y2,target_x0:target_x2] += 1
        return output / weight.clamp_min(1)

    def run_with_oom_retry(self, fn):
        try:
            return fn()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and self.tile_size > 128:
                print(f'⚠️ CUDA OOM in {self.name}; retrying with smaller tile.')
                clear_memory(True)
                self.tile_size = max(128, self.tile_size // 2)
                return fn()
            raise

print('BaseUpscaler ready with tile inference, FP16 flag, OOM retry, and cleanup.')


# -------------------- merged from old cell 8 --------------------
class SpandrelUpscaler(BaseUpscaler):
    name = 'Spandrel'
    model_key = None

    def __init__(self, model_key='HAT', **kwargs):
        self.model_key = model_key
        meta = MODEL_REGISTRY[model_key]
        self.name = model_key
        self.supported_scales = meta['scales']
        self.license_info = {'code': meta['code_license'], 'weights': meta['weight_license']}
        super().__init__(**kwargs)
        self.meta = meta

    def load(self):
        ensure_spandrel(extra_arches=False)
        from spandrel import ImageModelDescriptor, ModelLoader
        if self.scale not in self.meta['hf_files']:
            raise UpscalerError(f'{self.name} has no registered {self.scale}x weight.')
        weight_path = hf_download(self.meta['hf_repo'], self.meta['hf_files'][self.scale], min_bytes=50*1024*1024)
        descriptor = ModelLoader().load_from_file(str(weight_path))
        if not isinstance(descriptor, ImageModelDescriptor):
            raise UpscalerError(f'{self.name} checkpoint did not load as an image SR model via Spandrel.')
        descriptor = descriptor.to(self.device).eval()
        if self.fp16 and getattr(descriptor, 'supports_half', True):
            descriptor = descriptor.half()
        self.model = descriptor
        self.native_scale = int(getattr(descriptor, 'scale', self.scale) or self.scale)
        return self

    def upscale(self, input_path, output_path):
        if self.model is None:
            self.load()
        img, _ = load_image_rgb(input_path)
        x = self.pil_to_tensor(img).to(self.device)
        if self.fp16:
            x = x.half()
        def infer():
            y = self.forward_tiled(self.model, x, scale=self.native_scale)
            return y
        y = self.run_with_oom_retry(infer)
        out = self.tensor_to_pil(y)
        if self.native_scale != self.scale:
            out = out.resize((img.width*self.scale, img.height*self.scale), Image.Resampling.LANCZOS)
            self.notes.append(f'Native {self.native_scale}x output resized to requested {self.scale}x.')
        out.save(output_path)
        return Path(output_path)

class HATUpscaler(SpandrelUpscaler):
    def __init__(self, **kwargs): super().__init__(model_key='HAT', **kwargs)

class RealHATUpscaler(SpandrelUpscaler):
    def __init__(self, **kwargs): super().__init__(model_key='Real-HAT', **kwargs)

print('HAT and Real-HAT classes ready (Spandrel, lazy weight download, one model in VRAM at a time).')


# -------------------- merged from old cell 9 --------------------
import importlib.util

class SwinIRUpscaler(BaseUpscaler):
    name = 'SwinIR'
    supported_scales = [2,4]
    license_info = {'code': 'SwinIR Apache-2.0', 'weights': 'Official GitHub release; verify terms'}

    def _ensure_arch(self):
        ensure_package('timm', 'timm', optional=False)
        arch_path = CACHE_DIR / 'swinir' / 'network_swinir.py'
        if not arch_path.exists():
            arch_path.parent.mkdir(parents=True, exist_ok=True)
            download_url(MODEL_REGISTRY['SwinIR']['arch_url'], arch_path, min_bytes=10_000)
        spec = importlib.util.spec_from_file_location('network_swinir_colab', str(arch_path))
        mod = importlib.util.module_from_spec(spec)
        sys.modules['network_swinir_colab'] = mod
        spec.loader.exec_module(mod)
        return mod

    def load(self):
        mod = self._ensure_arch()
        weight = download_url(MODEL_REGISTRY['SwinIR']['weights'][4], MODEL_DIR/'SwinIR_x4.pth', min_bytes=10_000_000)
        model = mod.SwinIR(
            upscale=4, in_chans=3, img_size=64, window_size=8, img_range=1.,
            depths=[6,6,6,6,6,6], embed_dim=180, num_heads=[6,6,6,6,6,6],
            mlp_ratio=2, upsampler='nearest+conv', resi_connection='1conv'
        )
        ckpt = torch_load_weights_only(weight, map_location='cpu')
        state = ckpt.get('params_ema') or ckpt.get('params') or ckpt
        model.load_state_dict(state, strict=True)
        model.eval().to(self.device)
        if self.fp16:
            model = model.half()
        self.model = model
        self.native_scale = 4
        if self.scale == 2:
            self.notes.append('SwinIR official real-world checkpoint is native 4x; output will be resized down to 2x for requested scale.')
        return self

    def _forward_padded(self, x):
        # Official SwinIR expects H/W compatible with window_size. Pad each full image
        # or tile patch to a multiple of 8, then crop back to exact native output size.
        window = 8
        h, w = x.shape[-2:]
        pad_h = (window - h % window) % window
        pad_w = (window - w % window) % window
        if pad_h or pad_w:
            x_in = F.pad(x, (0, pad_w, 0, pad_h), mode='reflect')
        else:
            x_in = x
        y = self.model(x_in)
        return y[..., :h*4, :w*4]

    def upscale(self, input_path, output_path):
        if self.model is None:
            self.load()
        img, _ = load_image_rgb(input_path)
        x = self.pil_to_tensor(img).to(self.device)
        if self.fp16: x = x.half()
        def infer():
            return self.forward_tiled(self._forward_padded, x, scale=4)
        y = self.run_with_oom_retry(infer)
        out = self.tensor_to_pil(y)
        if self.scale != 4:
            out = out.resize((img.width*self.scale, img.height*self.scale), Image.Resampling.LANCZOS)
        out.save(output_path)
        return Path(output_path)

print('SwinIR class ready (official architecture + official x4 real-world checkpoint).')


# -------------------- merged from old cell 10 --------------------
class ResidualDenseBlock(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2*gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3*gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4*gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(nf, gc)
        self.rdb2 = ResidualDenseBlock(nf, gc)
        self.rdb3 = ResidualDenseBlock(nf, gc)
    def forward(self, x):
        return self.rdb3(self.rdb2(self.rdb1(x))) * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, scale=4, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.scale = int(scale)
        # RealESRGAN_x2plus uses pixel-unshuffle before RRDB: input channels become 12.
        # RealESRGAN_x4plus uses normal RGB input channels. Both variants then use two
        # nearest-neighbor upsample blocks; x2 starts from half-resolution features.
        conv_first_in_ch = 3 * 4 if self.scale == 2 else 3
        self.conv_first = nn.Conv2d(conv_first_in_ch, num_feat, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, 3, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    def forward(self, x):
        out_h, out_w = x.shape[-2] * self.scale, x.shape[-1] * self.scale
        if self.scale == 2:
            # F.pixel_unshuffle requires even H/W. Pad safely, then crop final output.
            pad_h = x.shape[-2] % 2
            pad_w = x.shape[-1] % 2
            if pad_h or pad_w:
                x = F.pad(x, (0, pad_w, 0, pad_h), mode='reflect')
            x = F.pixel_unshuffle(x, downscale_factor=2)
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode='nearest')))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode='nearest')))
        feat = self.lrelu(self.conv_hr(feat))
        out = self.conv_last(feat).clamp(0,1)
        return out[..., :out_h, :out_w]

class RealESRGANUpscaler(BaseUpscaler):
    name = 'Real-ESRGAN'
    supported_scales = [2,4]
    license_info = {'code': 'Real-ESRGAN project license; internal RRDB inference here', 'weights': 'Official Real-ESRGAN release weights'}

    def load(self):
        url = MODEL_REGISTRY['Real-ESRGAN']['weights'][self.scale]
        expected = MODEL_REGISTRY['Real-ESRGAN']['expected_bytes'].get(self.scale)
        weight = download_url(url, MODEL_DIR/f'RealESRGAN_x{self.scale}plus.pth', expected_bytes=expected)
        model = RRDBNet(scale=self.scale, num_feat=64, num_block=23, num_grow_ch=32)
        ckpt = torch_load_weights_only(weight, map_location='cpu')
        state = ckpt.get('params_ema') or ckpt.get('params') or ckpt
        cleaned = {k.replace('module.',''): v for k,v in state.items()}
        model.load_state_dict(cleaned, strict=True)
        model.eval().to(self.device)
        if self.fp16:
            model = model.half()
        self.model = model
        self.native_scale = self.scale
        return self

    def upscale(self, input_path, output_path):
        if self.model is None:
            self.load()
        img, _ = load_image_rgb(input_path)
        x = self.pil_to_tensor(img).to(self.device)
        if self.fp16: x = x.half()
        def infer():
            return self.forward_tiled(self.model, x, scale=self.scale)
        y = self.run_with_oom_retry(infer)
        out = self.tensor_to_pil(y)
        out.save(output_path)
        return Path(output_path)

UPSCALER_CLASSES = {
    'HAT': HATUpscaler,
    'Real-HAT': RealHATUpscaler,
    'SwinIR': SwinIRUpscaler,
    'Real-ESRGAN': RealESRGANUpscaler,
}
print('Real-ESRGAN internal RRDB class ready. Registered upscalers:', list(UPSCALER_CLASSES))



In [ ]:
# CELL 4 — Router, Quality Gate, Benchmark Mode, Auto Mode
#@title CELL 4 — Router, Quality Gate, Benchmark Mode, Auto Mode


# -------------------- merged from old cell 13 --------------------
class RuleBasedRouter:
    def __init__(self, registry):
        self.registry = registry

    def route(self, features, scale=4, available_models=None):
        available_models = available_models or list(UPSCALER_CLASSES.keys())
        flags = features['flags']
        scores = {m: 0.0 for m in available_models if scale in MODEL_REGISTRY.get(m,{}).get('scales', [])}
        reasons = {m: [] for m in scores}
        if not scores:
            return {'selected_model': None, 'confidence': 0.0, 'reason': 'No model supports requested scale.', 'fallback_model': None, 'ranked': []}
        # Extensible weighted rules.
        if flags['high_detail'] and not flags['noisy'] and not flags['compressed']:
            for m,w in [('HAT',0.38),('SwinIR',0.22),('Real-HAT',0.18)]:
                if m in scores: scores[m]+=w; reasons[m].append('high detail + low degradation')
        if flags['noisy'] or flags['compressed'] or flags['blurry']:
            for m,w in [('Real-ESRGAN',0.42),('SwinIR',0.22),('Real-HAT',0.14)]:
                if m in scores: scores[m]+=w; reasons[m].append('noise/compression/blur degradation')
        # Natural photo conservative path.
        if 0.16 <= features['contrast'] <= 0.55 and 0.18 <= features['brightness'] <= 0.88:
            for m,w in [('SwinIR',0.30),('HAT',0.18),('Real-ESRGAN',0.10)]:
                if m in scores: scores[m]+=w; reasons[m].append('natural photo tone range')
        if features['complexity'] > 0.55:
            for m,w in [('HAT',0.22),('Real-HAT',0.18),('SwinIR',0.12)]:
                if m in scores: scores[m]+=w; reasons[m].append('high complexity/detail')
        if scale == 2:
            if 'Real-HAT' in scores: scores.pop('Real-HAT', None)
            if 'SwinIR' in scores:
                scores['SwinIR'] -= 0.08; reasons['SwinIR'].append('2x uses x4 native then downsample')
        # Small baseline so every available model can be fallback.
        for m in scores: scores[m] += 0.05
        ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
        best, best_score = ranked[0]
        second = ranked[1][0] if len(ranked) > 1 else None
        total = sum(max(v,0) for _,v in ranked) + 1e-6
        confidence = max(0.0, min(1.0, best_score/total + 0.25))
        return {
            'selected_model': best,
            'confidence': round(confidence, 4),
            'reason': '; '.join(reasons.get(best, [])) or 'best available default from rule-based router',
            'fallback_model': second,
            'ranked': [{'model':m, 'router_score':round(float(v),5), 'reasons':reasons.get(m, [])} for m,v in ranked],
        }

ROUTER = RuleBasedRouter(MODEL_REGISTRY)
print('Rule-based router ready. It returns selected_model, confidence, reason, fallback_model, ranked candidates.')


# -------------------- merged from old cell 14 --------------------
def safe_metric(v, default=None):
    return default if v is None or (isinstance(v,float) and not math.isfinite(v)) else v

def score_candidate(metrics, artifacts, conservative=False):
    # Higher is better for MUSIQ, sharpness/detail/edge preservation. Lower is better for NIQE/BRISQUE/LPIPS/artifacts.
    musiq = safe_metric(metrics.get('MUSIQ'))
    niqe = safe_metric(metrics.get('NIQE'))
    brisque = safe_metric(metrics.get('BRISQUE'))
    lp = safe_metric(metrics.get('LPIPS_to_input_resized'))
    sharp = safe_metric(metrics.get('sharpness'), 0)
    tex = safe_metric(metrics.get('texture_density'), 0)
    edge = safe_metric(metrics.get('edge_density'), 0)
    quality_score = 0.0
    weights = 0.0
    if musiq is not None:
        quality_score += norm01(musiq, 35, 75) * 0.45; weights += 0.45
    if niqe is not None:
        quality_score += norm01(niqe, 2.5, 8.5, invert=True) * 0.28; weights += 0.28
    if brisque is not None:
        quality_score += norm01(brisque, 10, 70, invert=True) * 0.27; weights += 0.27
    if weights == 0:
        quality_score = 0.5
    else:
        quality_score /= weights
    fidelity_score = 0.65 if lp is None else norm01(lp, 0.08, 0.55, invert=True)
    detail_score = 0.55*norm01(sharp, 40, 900) + 0.25*norm01(tex, 0.12, 0.55) + 0.20*norm01(edge, 0.015, 0.16)
    artifact_penalty = artifacts.get('artifact_score', 0.0)
    if conservative:
        # Adobe Stock conservative mode: fidelity and low artifact risk beat raw sharpness.
        final = 100.0 * (0.40*quality_score + 0.38*fidelity_score + 0.22*detail_score - 0.58*artifact_penalty)
    else:
        final = 100.0 * (0.42*quality_score + 0.28*fidelity_score + 0.30*detail_score - 0.38*artifact_penalty)
    final = max(0.0, min(100.0, final))
    return {
        'quality_score': round(float(quality_score*100),4),
        'fidelity_score': round(float(fidelity_score*100),4),
        'detail_score': round(float(detail_score*100),4),
        'artifact_penalty': round(float(artifact_penalty*100),4),
        'final_score': round(float(final),4),
    }

def quality_gate(scores, artifacts, adobe_stock_mode=True):
    warnings_list = list(artifacts.get('warnings', []))
    fail_reasons = []
    min_score = 62 if adobe_stock_mode else 55
    max_artifact = 0.32 if adobe_stock_mode else 0.42
    if scores['final_score'] < min_score:
        fail_reasons.append(f'final_score below {min_score}')
    if artifacts.get('artifact_score', 0) > max_artifact:
        fail_reasons.append('artifact_score too high')
    if adobe_stock_mode:
        for risky in ['fake_texture','repeated_texture','face_distortion_risk','text_distortion_risk','color_artifacts','halos','ringing']:
            if artifacts.get('scores',{}).get(risky,0) > 0.50:
                fail_reasons.append(f'Adobe Stock risk: {risky}')
    status = 'FAIL' if fail_reasons else ('WARNING' if warnings_list or scores['final_score'] < 68 else 'PASS')
    return {'status': status, 'fail_reasons': fail_reasons, 'warnings': warnings_list, 'disclaimer': 'Technical QC only — platform acceptance is not guaranteed.'}

print('Quality gate ready. It combines quality + fidelity + detail - artifact penalty and never relies on one metric only.')


# -------------------- merged from old cell 15 --------------------
BENCHMARK_RESULTS = []
RUN_CONTEXT = {}

def run_single_candidate(model_name, input_path, analysis, scale, tile_size, tile_pad, batch_size, fp16, output_stem):
    result_path = OUTPUT_DIR / f'{output_stem}_{model_name.replace("-","_")}.png'
    notes, errors = [], []
    t0 = time.time()
    peak_mem = None
    try:
        if model_name not in UPSCALER_CLASSES:
            raise UpscalerError(f'{model_name} is not implemented.')
        cls = UPSCALER_CLASSES[model_name]
        up = cls(scale=scale, tile_size=tile_size, tile_pad=tile_pad, batch_size=batch_size, fp16=fp16)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        up.upscale(input_path, result_path)
        notes.extend(up.notes)
        if torch.cuda.is_available():
            peak_mem = round(torch.cuda.max_memory_allocated()/(1024**3), 4)
        up.unload()
        proc = round(time.time()-t0, 3)
        metrics = compute_quality_metrics(input_path, result_path, RUN_CONTEXT.get('ground_truth_path') or None)
        artifacts = detect_artifacts(input_path, result_path)
        conservative = bool(RUN_CONTEXT.get('adobe_stock_safety', True))
        scores = score_candidate(metrics, artifacts, conservative=conservative)
        gate = quality_gate(scores, artifacts, adobe_stock_mode=conservative)
        scores.update({'gate_status': gate['status'], 'gate_fail_reasons': gate['fail_reasons']})
        notes.extend(gate['warnings'])
        return CandidateResult(model_name, str(result_path), False, scale, {
            'tile_size': tile_size, 'tile_pad': tile_pad, 'batch_size': batch_size, 'fp16': fp16, 'peak_vram_gb': peak_mem,
        }, analysis, metrics, artifacts, scores, proc, HARDWARE.get('gpu_name','CPU'), notes, errors)
    except Exception as e:
        clear_memory(True)
        errors.append(str(e))
        return CandidateResult(model_name, '', False, scale, {
            'tile_size': tile_size, 'tile_pad': tile_pad, 'batch_size': batch_size, 'fp16': fp16, 'peak_vram_gb': peak_mem,
        }, analysis, {}, {}, {'final_score': 0, 'gate_status': 'ERROR'}, round(time.time()-t0,3), HARDWARE.get('gpu_name','CPU'), notes, errors)

def run_benchmark(input_path, analysis, scale=4, candidate_models=None, tile_size=None, tile_pad=None, batch_size=1, fp16=True):
    candidate_models = candidate_models or ['HAT','SwinIR','Real-ESRGAN']
    results = []
    for m in candidate_models:
        if scale not in MODEL_REGISTRY.get(m,{}).get('scales',[]):
            print(f'⚠️ Skipping {m}: no {scale}x support in registry.')
            continue
        print(f'\n===== Benchmark candidate: {m} =====')
        res = run_single_candidate(m, input_path, analysis, scale, tile_size or AUTO_TILE_SIZE, tile_pad if tile_pad is not None else AUTO_TILE_PAD, batch_size, fp16, 'candidate')
        if res.errors:
            print('❌', m, res.errors[-1])
        else:
            print('✅', m, 'final_score=', res.scores.get('final_score'), 'gate=', res.scores.get('gate_status'), 'time=', res.processing_time)
        results.append(res)
        clear_memory(True)
    valid = [r for r in results if not r.errors and r.output_path]
    if valid:
        best = sorted(valid, key=lambda r: r.scores.get('final_score',0), reverse=True)[0]
        for r in results:
            r.selected = (r.model == best.model)
        shutil.copy(best.output_path, OUTPUT_DIR/'final.png')
        print('Selected best benchmark output:', best.model)
    return results

print('Benchmark mode ready: runs candidates sequentially, scores every candidate, selects best valid output.')


# -------------------- merged from old cell 16 --------------------
AUTO_RESULTS = []

def run_auto(input_path, analysis, scale=4, tile_size=None, tile_pad=None, batch_size=1, fp16=True):
    route = ROUTER.route(analysis, scale=scale, available_models=list(UPSCALER_CLASSES.keys()))
    RUN_CONTEXT['router'] = route
    print('Router decision:', json.dumps(route, indent=2))
    candidates = []
    for item in route.get('ranked', [])[:3]:
        m = item['model']
        if m not in [c.model for c in candidates]:
            candidates.append(m)
    if route.get('fallback_model') and route['fallback_model'] not in candidates:
        candidates.append(route['fallback_model'])
    results = []
    for idx, m in enumerate(candidates):
        print(f'\n===== Auto candidate {idx+1}: {m} =====')
        res = run_single_candidate(m, input_path, analysis, scale, tile_size or AUTO_TILE_SIZE, tile_pad if tile_pad is not None else AUTO_TILE_PAD, batch_size, fp16, f'auto_{idx+1}')
        results.append(res)
        if not res.errors:
            print('Gate:', res.scores.get('gate_status'), 'Score:', res.scores.get('final_score'))
            if res.scores.get('gate_status') == 'PASS':
                print('Quality gate PASS. Stopping auto fallback chain.')
                break
        else:
            print('❌ Candidate failed:', res.errors[-1])
        clear_memory(True)
    valid = [r for r in results if not r.errors and r.output_path]
    if valid:
        passing = [r for r in valid if r.scores.get('gate_status') == 'PASS']
        selected = sorted(passing or valid, key=lambda r: r.scores.get('final_score',0), reverse=True)[0]
        for r in results:
            r.selected = (r.model == selected.model and r.output_path == selected.output_path)
        shutil.copy(selected.output_path, OUTPUT_DIR/'final.png')
        if not passing:
            print('⚠️ No candidate passed Quality Gate. Returning best available candidate with warnings.')
        print('Selected output:', selected.model, selected.output_path)
    else:
        raise RuntimeError('All auto candidates failed. See errors in report.')
    return results

print('Auto mode ready: analyzer → router → primary model → quality gate → fallback → best candidate.')



In [ ]:
# CELL 5 — Upload + Run Pipeline
#@title CELL 5 — Upload + Run Pipeline


# -------------------- merged from old cell 17 --------------------
MODE = "AUTO" #@param ["AUTO", "BENCHMARK"]
SCALE = "4x" #@param ["2x", "4x"]
TILE_SIZE = 0 #@param {type:"integer"}
TILE_PAD = -1 #@param {type:"integer"}
BATCH_SIZE = 1 #@param {type:"integer"}
USE_FP16 = True #@param {type:"boolean"}
BENCHMARK_MODELS = 'HAT,SwinIR,Real-ESRGAN' #@param {type:"string"}
ADOBE_STOCK_SAFETY = True #@param {type:"boolean"}
UPLOAD_NOW = True #@param {type:"boolean"}
GROUND_TRUTH_PATH = '' #@param {type:"string"}

INPUT_IMAGE = None
IMAGE_ANALYSIS = None
PIPELINE_RESULTS = []

def upload_image():
    if not IN_COLAB:
        raise RuntimeError('This upload interface is designed for Google Colab. Set INPUT_IMAGE manually when running locally.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')
    name, data = next(iter(uploaded.items()))
    src = INPUT_DIR / Path(name).name
    with open(src, 'wb') as f:
        f.write(data)
    return src

if UPLOAD_NOW:
    INPUT_IMAGE = upload_image()
    print('Uploaded:', INPUT_IMAGE)
    IMAGE_ANALYSIS = analyze_image(INPUT_IMAGE)
    RUN_CONTEXT.update({
        'input_image': str(INPUT_IMAGE),
        'mode': MODE,
        'scale': int(SCALE.replace('x','')),
        'tile_size': int(TILE_SIZE) if int(TILE_SIZE) > 0 else AUTO_TILE_SIZE,
        'tile_pad': int(TILE_PAD) if int(TILE_PAD) >= 0 else AUTO_TILE_PAD,
        'batch_size': int(BATCH_SIZE),
        'fp16': bool(USE_FP16 and torch.cuda.is_available()),
        'adobe_stock_safety': bool(ADOBE_STOCK_SAFETY),
        'ground_truth_path': GROUND_TRUTH_PATH.strip() or None,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    })
    display(Markdown('## Image Analysis'))
    print(json.dumps(IMAGE_ANALYSIS, indent=2))
    if RUN_CONTEXT.get('ground_truth_path'):
        print('Ground truth metrics enabled:', RUN_CONTEXT['ground_truth_path'])
    display(Image.open(INPUT_IMAGE))
else:
    print('UPLOAD_NOW is False. Set INPUT_IMAGE and IMAGE_ANALYSIS manually before running AUTO/BENCHMARK.')

START_PIPELINE = True #@param {type:"boolean"}
if START_PIPELINE and INPUT_IMAGE:
    scale = RUN_CONTEXT['scale']
    tile_size = RUN_CONTEXT['tile_size']
    tile_pad = RUN_CONTEXT['tile_pad']
    batch_size = RUN_CONTEXT['batch_size']
    fp16 = RUN_CONTEXT['fp16']
    if MODE == 'BENCHMARK':
        models = [m.strip() for m in BENCHMARK_MODELS.split(',') if m.strip()]
        PIPELINE_RESULTS = run_benchmark(INPUT_IMAGE, IMAGE_ANALYSIS, scale, models, tile_size, tile_pad, batch_size, fp16)
    else:
        PIPELINE_RESULTS = run_auto(INPUT_IMAGE, IMAGE_ANALYSIS, scale, tile_size, tile_pad, batch_size, fp16)
    BENCHMARK_RESULTS = PIPELINE_RESULTS if MODE == 'BENCHMARK' else []
    AUTO_RESULTS = PIPELINE_RESULTS if MODE == 'AUTO' else []
else:
    print('Pipeline not started yet.')



In [ ]:
# CELL 6 — Preview, Report, Download, Cleanup
#@title CELL 6 — Preview, Report, Download, Cleanup


# -------------------- merged from old cell 18 --------------------
def important_crop_box(path, crop_frac=0.32):
    rgb = image_to_np_rgb(path, max_side=None)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 180)
    h,w = gray.shape
    cw,ch = int(w*crop_frac), int(h*crop_frac)
    if cw < 64 or ch < 64:
        return (0,0,w,h)
    best = (0,0,-1)
    step_x = max(16, cw//3); step_y = max(16, ch//3)
    for y in range(0, h-ch+1, step_y):
        for x in range(0, w-cw+1, step_x):
            score = edges[y:y+ch, x:x+cw].mean() + gray[y:y+ch, x:x+cw].std()*0.02
            if score > best[2]: best = (x,y,score)
    x,y,_ = best
    return (x,y,x+cw,y+ch)

def make_comparison(results, input_path, out_path=OUTPUT_DIR/'comparison.png'):
    valid = [r for r in results if r.output_path and not r.errors]
    if not valid:
        print('No valid outputs for comparison.')
        return None
    imgs = [('Original', Image.open(input_path).convert('RGB'))]
    for r in valid:
        label = f"{r.model}\nscore {r.scores.get('final_score')}\n{r.scores.get('gate_status')}"
        imgs.append((label, Image.open(r.output_path).convert('RGB')))
    thumb_h = 420
    thumbs = []
    for label,img in imgs:
        im = img.copy()
        im.thumbnail((520, thumb_h), Image.Resampling.LANCZOS)
        thumbs.append((label, im))
    pad = 20; label_h = 80
    total_w = sum(im.width for _,im in thumbs) + pad*(len(thumbs)+1)
    total_h = thumb_h + label_h + 260
    canvas = Image.new('RGB', (total_w, total_h), (245,245,245))
    draw = ImageDraw.Draw(canvas)
    x = pad
    for label, im in thumbs:
        canvas.paste(im, (x, label_h))
        draw.text((x, 10), label, fill=(20,20,20))
        x += im.width + pad
    # Detail crop comparison
    box = important_crop_box(input_path)
    draw.text((pad, thumb_h+label_h+20), 'Detail crop comparison', fill=(0,0,0))
    x = pad
    crop_y = thumb_h + label_h + 50
    for label, img in imgs:
        src = img
        # map original crop to each output size by scale ratio
        if label != 'Original':
            ow, oh = Image.open(input_path).size
            sx, sy = src.width/ow, src.height/oh
            b = tuple(int(v*sx) if i%2==0 else int(v*sy) for i,v in enumerate(box))
        else:
            b = box
        crop = src.crop(b)
        crop.thumbnail((220, 180), Image.Resampling.LANCZOS)
        canvas.paste(crop, (x, crop_y))
        draw.text((x, crop_y+crop.height+4), label.split('\n')[0], fill=(20,20,20))
        x += 240
    canvas.save(out_path)
    return out_path

if 'PIPELINE_RESULTS' in globals() and PIPELINE_RESULTS:
    comp = make_comparison(PIPELINE_RESULTS, INPUT_IMAGE)
    if comp:
        display(Image.open(comp))
        print('Comparison saved:', comp)
    final = OUTPUT_DIR/'final.png'
    if final.exists():
        display(Markdown('## Final Output'))
        display(Image.open(final))
else:
    print('No pipeline results yet. Run CELL 17 first.')


# -------------------- merged from old cell 19 --------------------
def result_to_dict(r):
    d = asdict(r)
    return d

def export_reports(results, input_path=None):
    rows = []
    for r in results:
        rows.append({
            'input': str(input_path) if input_path else RUN_CONTEXT.get('input_image'),
            'model': r.model,
            'scale': r.scale,
            'selected': r.selected,
            'processing_time': r.processing_time,
            'GPU': r.gpu,
            'MUSIQ': r.metrics.get('MUSIQ'),
            'NIQE': r.metrics.get('NIQE'),
            'BRISQUE': r.metrics.get('BRISQUE'),
            'LPIPS': r.metrics.get('LPIPS_to_input_resized'),
            'sharpness': r.metrics.get('sharpness'),
            'edge_density': r.metrics.get('edge_density'),
            'artifact_score': r.artifacts.get('artifact_score'),
            'final_score': r.scores.get('final_score'),
            'gate_status': r.scores.get('gate_status'),
            'errors': ' | '.join(r.errors),
            'output_path': r.output_path,
        })
    report = {
        'README': {
            'architecture': 'Input Image → Image Analyzer → Model Router → Candidate Upscaling → Quality Evaluation → Artifact Detection → Best Output → Report',
            'models': MODEL_REGISTRY,
            'model_provenance': {k: v.get('provenance', {}) for k, v in MODEL_REGISTRY.items()},
            'usage': 'Run cells 1-17, choose AUTO or BENCHMARK, then cells 18-20.',
            'benchmark': 'BENCHMARK mode runs selected models sequentially and compares metrics/artifacts/final_score.',
            'limitations': 'No metric guarantees stock acceptance. Face/text/semantic distortion checks are conservative CV heuristics.',
            'adobe_stock_considerations': 'Prioritize fidelity; warnings flag hallucinated texture, distorted face/text, halos, ringing, excessive sharpening. Technical QC only — platform acceptance is not guaranteed.',
        },
        'context': RUN_CONTEXT,
        'input': str(input_path) if input_path else RUN_CONTEXT.get('input_image'),
        'analysis': IMAGE_ANALYSIS if 'IMAGE_ANALYSIS' in globals() else None,
        'router': RUN_CONTEXT.get('router'),
        'selected_model': next((r.model for r in results if r.selected), None),
        'results': [result_to_dict(r) for r in results],
    }
    json_path = REPORT_DIR/'quality_report.json'
    csv_path = REPORT_DIR/'quality_report.csv'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print('Reports saved:', json_path, csv_path)
    display(pd.DataFrame(rows).sort_values('final_score', ascending=False, na_position='last'))
    return json_path, csv_path

if 'PIPELINE_RESULTS' in globals() and PIPELINE_RESULTS:
    export_reports(PIPELINE_RESULTS, INPUT_IMAGE)
else:
    print('No results to export yet.')


# -------------------- merged from old cell 20 --------------------
CREATE_ZIP = True #@param {type:"boolean"}
AUTO_DOWNLOAD = False #@param {type:"boolean"}

clear_memory(True)
if CREATE_ZIP:
    zip_path = WORK_DIR/'open_sr_pipeline_results.zip'
    if zip_path.exists(): zip_path.unlink()
    import zipfile
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT_DIR.glob('*'):
            if p.is_file():
                z.write(p, arcname=f'output/{p.name}')
    print('ZIP created:', zip_path)
    if AUTO_DOWNLOAD and IN_COLAB:
        files.download(str(zip_path))
print('Cleanup complete. Models are not kept in VRAM between candidates.')

